In [1]:

import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

# 1. Load the Iris dataset
iris = load_iris()
X = iris.data
y = iris.target

# 2. Normalize input features
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Convert labels into one-hot encoding
Y = np.eye(3)[y]

# Split data into training and testing sets
X_train, X_test, Y_train, Y_test, y_train, y_test = train_test_split(
    X, Y, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Initialize neural network parameters
np.random.seed(42)

input_neurons = 4
hidden_neurons = 8
output_neurons = 3

W1 = np.random.randn(input_neurons, hidden_neurons) * 0.1
b1 = np.zeros((1, hidden_neurons))

W2 = np.random.randn(hidden_neurons, output_neurons) * 0.1
b2 = np.zeros((1, output_neurons))

learning_rate = 0.05
epochs = 3000

# Activation functions
def sigmoid(x):
    x = np.clip(x, -500, 500)
    return 1 / (1 + np.exp(-x))

def softmax(x):
    x = x - np.max(x, axis=1, keepdims=True)
    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

# 4. Train ANN using backpropagation
for epoch in range(epochs):

    # Forward propagation
    z1 = X_train @ W1 + b1
    a1 = sigmoid(z1)

    z2 = a1 @ W2 + b2
    a2 = softmax(z2)

    # Calculate cross-entropy loss
    loss = -np.mean(
        np.sum(Y_train * np.log(a2 + 1e-12), axis=1)
    )

    # Backpropagation: output layer
    dz2 = (a2 - Y_train) / len(X_train)
    dW2 = a1.T @ dz2
    db2 = np.sum(dz2, axis=0, keepdims=True)

    # Backpropagation: hidden layer
    dz1 = (dz2 @ W2.T) * a1 * (1 - a1)
    dW1 = X_train.T @ dz1
    db1 = np.sum(dz1, axis=0, keepdims=True)

    # Update weights and biases
    W2 -= learning_rate * dW2
    b2 -= learning_rate * db2
    W1 -= learning_rate * dW1
    b1 -= learning_rate * db1

    if (epoch + 1) % 500 == 0:
        print(f"Epoch {epoch + 1}, Loss: {loss:.4f}")

# 5. Test the trained ANN
hidden_output = sigmoid(X_test @ W1 + b1)
predicted_probabilities = softmax(hidden_output @ W2 + b2)
y_pred = np.argmax(predicted_probabilities, axis=1)

# 6. Display results
accuracy = accuracy_score(y_test, y_pred)

print("\nActual classes:   ", y_test)
print("Predicted classes:", y_pred)
print(f"\nTest Accuracy: {accuracy * 100:.2f}%")

print("\nClassification Report:")
print(classification_report(
    y_test, y_pred, target_names=iris.target_names
))


Epoch 500, Loss: 0.4707
Epoch 1000, Loss: 0.2935
Epoch 1500, Loss: 0.2055
Epoch 2000, Loss: 0.1531
Epoch 2500, Loss: 0.1224
Epoch 3000, Loss: 0.1036

Actual classes:    [0 2 1 1 0 1 0 0 2 1 2 2 2 1 0 0 0 1 1 2 0 2 1 2 2 1 1 0 2 0]
Predicted classes: [0 2 1 1 0 1 0 0 2 1 2 2 2 1 0 0 0 1 1 2 0 2 1 2 2 2 1 0 2 0]

Test Accuracy: 96.67%

Classification Report:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      0.90      0.95        10
   virginica       0.91      1.00      0.95        10

    accuracy                           0.97        30
   macro avg       0.97      0.97      0.97        30
weighted avg       0.97      0.97      0.97        30

